## B1 - Meet Your Camera
Author: George Gorospe, george.gorospe@nmaia.net\
Last Update: July 17th, 2026

About: In this notebook you'll learn about your robot's camera including how to capture images and what image data looks like.  

Almost all of the work we do next involves using the camera and using the images collected with the camera, so this is a very important notebook!

## Camera setup involves a common sequence of python commands, consider it as a template.
Step 1. Import libraries designed specifically for the camera, camera data, and visualizing the camera data (pictures).  
Step 2. Create a camera object that can be called to capture images.  
Step 3. Create jupyter laboratory widgets to visualize the camera data.  
Step 4. Link your camera object and the widget so that new images are automatically streamed.  
Step 5. Display the widgets. 

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# This cell imports all the libraries we'll use for this notebook

# Importing our custom library for our TraitletCamera
from jetcam_lite import TraitletCamera, bgr8_to_jpeg

# Importing our idempotency helper -- makes it safe to re-run
# cells that register button click handlers
from jupyter_utils import register_click_handler, register_dlink

# Importing graphical user interface libraries
import traitlets
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual, Layout
from IPython.display import display

# Importing computer vision and data management libraries
import numpy as np
import pandas as pd
import cv2

In [ ]:
### Camera Setup ###
# Step 2. Instantiate and start the camera
camera = TraitletCamera()
camera.start()

# Step 3. Create the widget to display the image feed
image_widget = widgets.Image(format='jpeg', width=camera.width, height=camera.height)

# Step 4. Link the widget and the camera feed
# register_dlink (instead of calling traitlets.dlink() directly) keeps this cell
# safe to re-run: it won't stack a second live link onto the widget.
register_dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

# Step 5. Display the image widget
display(image_widget)

### Creating Tools and Interfaces within our Notebooks Using Widgets:  
Jupyter notebooks started out with just two types of cells: markdown cells and code cells.  
But, people wanted more functions, so they created **widgets** aditional tools to add cool new features to notebooks.  
We're using some of those features here to visualize our camera images. (see #3 above)

To connect widgets together or with things like our camera, we're also using traitlets. (see #4 above)  
Traitlets act as a syncroniztion and notification tool.    

They are actually really useful!  
When a button widget is clicked, traitlets might run a function or update some other widget.  
When a new images is captured by the camera, the image widget is notified and receives the image after it is transformed from bgr8 to jpeg formats.


<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY B1.1: Capture a Single Frame</span>


Right now, `image_widget` shows a **live** feed — it updates on its own, many times a second. That's great for looking at, but not for working with: we need a way to freeze a single frame and hang onto it.

The cell below adds a **button**. Every time you click it, we'll grab whatever the camera sees at that exact instant and store it in a variable called `captured_image`.  
This is a new pattern: instead of code that just runs top to bottom, this code *waits* for you to click, then runs.  

### This pattern is called **event-driven programming**, we're going to use it a lot as we move forward.
Clicking a button is the event, this leads to the callback function, then the output of the function.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# This cell creates a "Capture Frame" button. Clicking it grabs the camera's
# CURRENT frame (always available as camera.value) displays it and saves it.

# TWO more widgets: a button and another display.
capture_button = widgets.Button(description='Capture Frame', button_style='success')
status_bar = widgets.Text(value='', placeholder='Waiting', description='Status:')


# This function runs every time the button is clicked.
def on_capture_clicked(button):
    global captured_image
    captured_image = camera.value
    status_bar.value = "Captured a frame!"

# Connect the button to the function above. When the button is clicked the function above is executed.
# register_click_handler (instead of calling .on_click() directly) keeps this cell
# safe to re-run: it won't stack a second copy of the handler onto the button.
register_click_handler(capture_button, on_capture_clicked)

# Displaying the user interface: image, button, status bar
display(image_widget) # The picture
display(widgets.HBox([capture_button, status_bar])) # Button and status bar

**Go ahead — click the button above!** Try moving something in front of the camera first, then click. You should see the status bar change to "Captured a frame!"  
You can click it again any time to grab a fresh frame — do that any time you want a new `captured_image` to work with in the activities below.

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY B1.2: What IS a Picture, Really?</span>


**Before you run the next cell, make a prediction with your group:**

 + What kind of Python object do you think `captured_image` actually is?
 + What do you think its **shape** (dimensions) will be?

Write down your guesses, then run the cell below to find out.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Let's find out! We'll examine 'captured_image' three ways:
# 1. What TYPE of object is it?
# 2. What SHAPE (dimensions) does it have?
# 3. What kind of numbers does it hold (its DTYPE)?

print("Type: ", type(captured_image))
print("Shape:", captured_image.shape)
print("Dtype:", captured_image.dtype)

**🔬 Real Data Science:** That picture you were just looking at is a **NumPy array** — a 3D grid of numbers. The shape tells you `(height, width, 3)`: a row/column position for every pixel, and *3* numbers at each one, for its color.

One detail worth knowing now, because it'll matter in the next activity: this camera stores color as **BGR** (Blue, Green, Red) — notice `bgr8_to_jpeg` in the import cell — not the RGB order you might expect. Different libraries make different choices here, and mixing them up is a classic bug.

This exact kind of array — a grid of numbers representing an image — is what every notebook this week will work with: capturing datasets, training models, and running them on the robot.

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY B1.3: Editing Pixels Directly</span>


If `captured_image` is just a grid of numbers, we should be able to *edit* it like any other array — and see the result. Below, you'll set a whole rectangle of pixels to a single color, directly.

**Remember:** colors here are **BGR**, not RGB! For example, pure red is `[0, 0, 255]` — zero blue, zero green, full red.

### **IMPORTANT** 👋 For this activity, go back and take a photo with your hand in the frame of the camera.

In [ ]:
#### ------> ACTIVITY B1.3: Editing Pixels Directly <-----#####
# Paint a solid colored square onto your finger tip in the captured image of your hand.

# Setup: copying and adding simple axis to the image.
edited_image = captured_image.copy()
cv2.putText(edited_image,"(0,0) Origin                                                    X-Axis  640", (0, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2, cv2.LINE_AA)
cv2.putText(edited_image,"Y-Axis", (5, 300), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
cv2.putText(edited_image,"400", (5, 400), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)


##### INSTRUCTIONS: #####
# 1. Choose one of the fingers in your image, you're going to draw a blue square on the finger tip.
# 2. Looking at the axis of the image try to identify the coordnates for your finger. The Y-axis goes from top to bottom and the X-axis goes left to right.
# 3. Adjust the X and Y coordinates below so that the that rectangle covers your finger tip.
# 4. Run this cell, then run the display cell below it to see your edit.

# Ajust this:
x_coordinate = 370
y_coordinate = 100

# Dont' adjust this:
box_size = 50 # length of each side (pixels)
end_point = [y_coordinate + box_size, x_coordinate + box_size] 
edited_image[y_coordinate:end_point[0], x_coordinate:end_point[1]] = [255,0,0]

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Display your edited captured_image (a single, frozen frame -- not the live feed).
# Shown larger (2x) here, since you're working precisely with pixels in this activity.

edited_widget = widgets.Image(format='jpeg', width=2*camera.width, height=2*camera.height)
edited_widget.value = bgr8_to_jpeg(edited_image)
display(edited_widget)

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY B1.4: Capture and Display a Frame from Live Feed</span>


### Expanding on our event-driven programming template with one button and two displays.
Starting with a plan: 
 + What we want is a display of the live feed, a display for the captured frame, and a button to capture a frame.
 + When the button is clicked a frame should be captured AND displayed on the second display.

What this requires:
 1. A live_feed display, let's call it ```live_feed_widget```
 2. A traitlet to connect the camera and the ```live_feed_widget```. It will notify the widget that new data is available.
 3. A new display for the captured image, we'll call it ```captured_image_widget```
 4. A button to press. We'll call it ```capture_button```
 5. A function to capture the image and display it on the ```captured_image_widget``
 6. A traitlet connection between the capture button and the function to actually capture the frame.

Look at the code in the next cell and add numbers and labels to each section in the comments.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Side-by-side: LIVE feed (left, keeps updating) vs. your CAPTURED frame (right, frozen).


# This is #?, it does ?
live_feed_widget = widgets.Image(format='jpeg', width=camera.width, height=camera.height)

# This is #?, it does ?
register_dlink((camera, 'value'), (live_feed_widget, 'value'), transform=bgr8_to_jpeg)

# This is #?, it does ?
captured_image_widget = widgets.Image(format='jpeg', width=camera.width, height=camera.height)

# This is #?, it does ?
capture_button = widgets.Button(description='Capture Frame', button_style='success')

# This is #?, it does ?
def capture_button_clicked(button):
    # Note: camera.value is a raw array -- bgr8_to_jpeg() converts it into the
    # encoded image bytes that an Image widget's .value actually expects.
    captured_image_widget.value = bgr8_to_jpeg(camera.value)

# This is #?, it does ?
register_click_handler(capture_button, capture_button_clicked)

display(live_feed_widget)
display(captured_image_widget)
display(capture_button)

<span style="color: green; font-size: 55px; font-style: italic;">Student Discussion Time</span>


## **STUDENT QUESTIONS**:
Talk these over with your group.

 + **Three numbers per pixel:** Why does a color image need three numbers per pixel instead of one? What do you think the shape of a black-and-white image would look like instead?

 + **BGR vs. RGB:** We just used Blue-Green-Red order instead of Red-Green-Blue. Why might it matter that different libraries expect colors in different orders? What would happen to a picture if you mixed them up?

 + **Live vs. captured:** The live feed updates on its own, but `captured_image` doesn't, unless you click the button again. Why do you think having a single, frozen frame -- rather than a constantly-changing stream -- matters for the machine learning work coming up this week?

 + **Lighting matters:** If two students captured pictures of the same object, but one had a bright window behind them and the other didn't, how would the resulting arrays differ? What problems could that cause later, when training a model?

### Now that you've completed this notebook, move on to `B2 - Capture Script and File Counting.ipynb`, where you'll turn this single-frame capture into a full dataset-collection tool.